# FT105A — Trabalho 1: PNAD Contínua (suporte de visualização)

Notebook de apoio ao relatório GRIVAPP. Base conceitual: **Tópico 04 — Dados multivariados** (Ward, Grinstein e Keim, 2010, conforme material da disciplina): técnicas classificadas pela primitiva gráfica — **ponto**, **linha**, **região** e combinações / **orientadas a pixel**.

Também entram diretrizes de **mapeamento visual** (Tópico 03): expressividade e efetividade ao ligar variáveis da tabela de dados a canais gráficos.

| Técnica | Família (Tópico 04) | No enunciado T1 |
| --- | --- | --- |
| Coordenadas paralelas | linha | candidata forte |
| Treemap | região / hierarquia (tópico árvores) | candidata forte |
| Orientada a pixels | dense pixel displays | candidata forte |
| Matriz de dispersão (SPLOM) | ponto + displays múltiplos | complementar |
| Parallel sets | linha (variante categórica) | complementar |
| RadViz | ponto / força | complementar |

O enunciado pede **três** visualizações com **três** técnicas distintas. As três primeiras formam o núcleo alinhado ao rascunho do grupo; as demais ampliam o repertório para escolha fundamentada.

**Dados:** `../../../data/processed/pnadc_2026q2_sample.parquet` · **Kernel:** `masters-curriculum`.


In [10]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "notebook_connected"

COURSE_DATA = Path("../../../data").resolve()
SAMPLE = COURSE_DATA / "processed" / "pnadc_2026q2_sample.parquet"

INSTR_ORDER = [
    "Sem instrucao",
    "Fund. incompleto ou equiv.",
    "Fund. completo ou equiv.",
    "Medio incompleto ou equiv.",
    "Medio completo ou equiv.",
    "Superior incompleto ou equiv.",
    "Superior completo",
]
COR_ORDER = ["Branca", "Preta", "Parda", "Amarela", "Indigena"]

df = pd.read_parquet(SAMPLE)
print(df.shape)
df.head()


(50000, 14)


,ano,trimestre,uf_cod,uf,sexo,idade,cor_raca,ocupacao_cbo,setor,rend_principal,instrucao,cond_ocupacao,rend_todos,peso
0,2026,2,41,PR,Homem,41,Branca,5222,Comercio e reparacao,2600.0,Fund. completo ou equiv.,Ocupado,2600.0,580.990103
1,2026,2,52,GO,Homem,50,Preta,9212,Agricultura,1200.0,Fund. incompleto ou equiv.,Ocupado,1200.0,96.464521
2,2026,2,35,SP,Mulher,41,Preta,4223,Admin publica,2000.0,Superior completo,Ocupado,2000.0,1362.102335
3,2026,2,42,SC,Homem,31,Branca,3341,Industria geral,7500.0,Superior completo,Ocupado,7500.0,370.628790
4,2026,2,27,AL,Homem,47,Parda,6111,Agricultura,1621.0,Fund. incompleto ou equiv.,Ocupado,1621.0,123.041093


## Julgamento das três técnicas-núcleo (para a PNAD)

Critérios alinhados ao material: (i) caber em **dados multivariados** com >3 variáveis; (ii) **mapeamento** legível (posição, cor, área); (iii) adequação a microdados socioeconômicos (mistura de contínuas e categóricas, n grande).

### Coordenadas paralelas — família **linha**

**Pontos fortes.** Codifica muitas dimensões sem reduzir o espaço a 2 eixos cartesianos; a tupla vira polilinha (Tópico 04). Na PNAD, idade, renda, instrução ordinal, UF e sexo cabem no mesmo display. Interação (filtro por eixo) ajuda a isolar faixas — paralelo ao “brushing” citado nas matrizes de dispersão.

**Limitações.** Sobrecarga visual com dezenas de milhares de linhas (overplotting); ordem dos eixos altera a leitura de vizinhanças; eixos categóricos precisam de ordenação explícita.

**Veredito.** Adequada como técnica central de associação entre atributos individuais.

### Treemap — família **região** / hierarquia

**Pontos fortes.** Área e encapsulamento (estrutura visual de parte–todo) comunicam magnitude e hierarquia UF → setor. Cor contínua (renda média) adiciona dimensão sem novo eixo. Escala bem com muitas células agregadas.

**Limitações.** Agrega indivíduos (não mostra cada pessoa); comparação fina de áreas médias é menos precisa que posição em eixo linear (efetividade relativa das marcas — Tópico 03).

**Veredito.** Boa para estrutura territorial/setorial da PNAD; complementa (não substitui) vistas no nível do indivíduo.

### Orientada a pixels — *dense pixel displays*

**Pontos fortes.** Maximiza quantos valores cabem na tela (Keim / material Tópico 04, § técnicas orientadas a pixel): um valor → um pixel, subjanelas por atributo, mesma ordenação. Natural para microdados densos (analogia às séries densas do exemplo de ações).

**Limitações.** Eixos cartesianos somem; a leitura é de textura e blocos de cor. Exige ordenação bem escolhida (aqui: renda).

**Veredito.** Adequada e fiel ao material — desde que **não** seja confundida com heatmap agregado.

### Síntese para o relatório

As três cobrem famílias distintas (linha, região, pixel) e escalas diferentes (indivíduo vs agregado vs densidade). Isso fortalece a argumentação de “três técnicas distintas” do enunciado.


## 1. Coordenadas paralelas (núcleo)

Cada dimensão vira eixo vertical; cada indivíduo, uma polilinha que cruza os eixos (Tópico 04, técnicas baseadas em linha). Agrupamentos de linhas parecidas sugerem correlação entre eixos vizinhos; linhas isoladas ou com curvatura atípica sugerem outliers.

**Mapeamento (PNAD):** eixos = idade, instrução ordinal, rendimento, UF, sexo; cor = sexo.

**Exemplo fora da PNAD:** perfis de alunos (idade, acessos, nota) com destaque de faixa de nota — como no rascunho do grupo.


In [11]:
def fig_parallel_coords(df: pd.DataFrame) -> go.Figure:
    d = df.loc[
        (df["cond_ocupacao"] == "Ocupado")
        & df["rend_todos"].notna() & (df["rend_todos"] > 0) & df["idade"].notna()
    ].copy()
    d = d.loc[d["rend_todos"] <= d["rend_todos"].quantile(0.99)]
    if len(d) > 8000:
        d = d.sample(n=8000, random_state=0)
    d["instr_ord"] = pd.Categorical(d["instrucao"], categories=INSTR_ORDER, ordered=True)
    d = d.dropna(subset=["instr_ord", "sexo"])
    d["instr_num"] = d["instr_ord"].cat.codes.astype(float)
    d["sexo_num"] = (d["sexo"] == "Mulher").astype(float)
    d["uf_num"] = pd.to_numeric(d["uf_cod"], errors="coerce")
    fig = go.Figure(data=go.Parcoords(
        line=dict(color=d["sexo_num"], colorscale=[[0, "#0072B2"], [1, "#D55E00"]],
                  showscale=True, colorbar=dict(title="Sexo", tickvals=[0, 1], ticktext=["Homem", "Mulher"])),
        dimensions=[
            dict(label="Idade", values=d["idade"], range=[14, float(d["idade"].max())]),
            dict(label="Instrucao (ord)", values=d["instr_num"],
                 tickvals=list(range(len(INSTR_ORDER))), ticktext=INSTR_ORDER),
            dict(label="Rend. todos (R$)", values=d["rend_todos"]),
            dict(label="UF cod.", values=d["uf_num"]),
            dict(label="Sexo (0/1)", values=d["sexo_num"], tickvals=[0, 1], ticktext=["H", "M"]),
        ],
        labelangle=-15,
    ))
    fig.update_layout(title=f"Coordenadas paralelas — ocupados (n={len(d):,})", height=560,
                      margin=dict(l=80, r=40, t=80, b=40), font=dict(size=12))
    return fig

fig1 = fig_parallel_coords(df)
fig1.show()


## 2. Treemap (núcleo)

Retângulos aninhados: área ∝ magnitude; hierarquia por encapsulamento (estruturas visuais / árvores). Na PNAD, o nível espacial (UF) contém setores; a cor leva a renda média da célula.

**Mapeamento:** path = UF → setor; área = contagem de ocupados; cor = renda média (Cividis).

**Exemplo fora da PNAD:** árvore de diretórios vs treemap de ocupação de disco.


In [12]:
def fig_treemap(df: pd.DataFrame) -> go.Figure:
    d = df.loc[
        (df["cond_ocupacao"] == "Ocupado") & df["setor"].notna()
        & (df["setor"] != "Nao classificado / NA")
        & df["rend_todos"].notna() & (df["rend_todos"] > 0)
    ].copy()
    g = d.groupby(["uf", "setor"], as_index=False).agg(n=("rend_todos", "size"), rend_medio=("rend_todos", "mean"))
    fig = px.treemap(g, path=["uf", "setor"], values="n", color="rend_medio", color_continuous_scale="Cividis",
                     title=f"Treemap UF → setor (area=n; cor=renda media); n_obs={len(d):,}")
    fig.update_traces(textinfo="label+value+percent root",
                      hovertemplate="<b>%{label}</b><br>n=%{value}<br>renda media=%{color:.0f}<extra></extra>")
    fig.update_layout(margin=dict(t=70, l=10, r=10, b=10), height=650)
    return fig

fig2 = fig_treemap(df)
fig2.show()


## 3. Técnica orientada a pixels (núcleo)

Conforme o Tópico 04 (*dense pixel displays* / técnicas orientadas a pixel): cada valor mapeia-se a um pixel; atributos ocupam subjanelas; a ordenação comum alinha o mesmo registro em todas as janelas. Não é heatmap de médias por célula categórica.

**Mapeamento:** ordem = renda crescente; subjanelas = log-renda, idade, flag ocupado, instrução ordinal; cor = Viridis.


In [13]:
def _to_grid(values: np.ndarray, n_cols: int) -> np.ndarray:
    n = len(values)
    n_rows = int(math.ceil(n / n_cols))
    grid = np.full((n_rows, n_cols), np.nan, dtype=float)
    grid.flat[:n] = values
    return grid

def fig_pixel_matrix(df: pd.DataFrame, n_max: int = 10_000, n_cols: int = 100) -> go.Figure:
    d = df.loc[df["rend_todos"].notna() & (df["rend_todos"] > 0) & df["idade"].notna() & df["instrucao"].notna()].copy()
    d["instr_ord"] = pd.Categorical(d["instrucao"], categories=INSTR_ORDER, ordered=True)
    d = d.dropna(subset=["instr_ord"])
    d["instr_num"] = d["instr_ord"].cat.codes.astype(float)
    d["log_rend"] = np.log1p(d["rend_todos"])
    d["ocupado"] = (d["cond_ocupacao"] == "Ocupado").astype(float)
    d = d.sort_values("rend_todos", kind="mergesort")
    if len(d) > n_max:
        d = d.iloc[np.linspace(0, len(d) - 1, n_max).astype(int)]
    attrs = [
        ("log1p(renda)", d["log_rend"].to_numpy()),
        ("idade", d["idade"].to_numpy()),
        ("ocupado (0/1)", d["ocupado"].to_numpy()),
        ("instrucao (ord)", d["instr_num"].to_numpy()),
    ]
    fig = make_subplots(rows=1, cols=len(attrs), subplot_titles=[a[0] for a in attrs], horizontal_spacing=0.06)
    for i, (label, vals) in enumerate(attrs, start=1):
        fig.add_trace(go.Heatmap(
            z=_to_grid(vals, n_cols), colorscale="Viridis", showscale=(i == len(attrs)),
            colorbar=dict(title="valor", len=0.7) if i == len(attrs) else None,
            hovertemplate="linha=%{y}<br>col=%{x}<br>valor=%{z:.3f}<extra>" + label + "</extra>",
        ), row=1, col=i)
        fig.update_xaxes(showticklabels=False, row=1, col=i)
        fig.update_yaxes(showticklabels=False, autorange="reversed", row=1, col=i)
    fig.update_layout(
        title=f"Dense pixel display — ordem por renda; n={len(d):,}; {n_cols} colunas",
        height=420, margin=dict(t=90, l=20, r=20, b=20),
    )
    return fig

fig3 = fig_pixel_matrix(df)
fig3.show()


## 4. Matriz de gráficos de dispersão (SPLOM) — complementar

### Ligação com o material

Tópico 04, §1.1: técnicas **baseadas em ponto**; a matriz de dispersão usa **displays múltiplos** / *small multiples*: N variáveis → grade N×N; cada célula é o scatter de um par (ou histograma/nome na diagonal). O material destaca ainda o **brushing** / visões coordenadas (marcar pontos em um painel reflete nos outros).

### Por que faz sentido na PNAD

Idade e log-renda são contínuas naturais para posição espacial (canal de alta efetividade). Cor e símbolo podem incorporar sexo e cor/raça — **incorporar dimensões** além do plano (slide “representando mais dimensões”).

### Julgamento

**Prós:** leitura familiar de nuvens e correlação linear local; forte no material da disciplina.  
**Contras:** número de painéis cresce com N²; categóricas demais exigem jitter ou não entram bem no eixo.

**Mapeamento abaixo:** idade × log-renda × instrução ordinal; cor = sexo; amostra de ocupados.


In [14]:
def fig_splom(df: pd.DataFrame, n_max: int = 4000) -> go.Figure:
    d = df.loc[
        (df["cond_ocupacao"] == "Ocupado")
        & df["rend_todos"].notna() & (df["rend_todos"] > 0)
        & df["idade"].notna() & df["instrucao"].notna() & df["sexo"].notna()
    ].copy()
    d = d.loc[d["rend_todos"] <= d["rend_todos"].quantile(0.99)]
    d["instr_ord"] = pd.Categorical(d["instrucao"], categories=INSTR_ORDER, ordered=True)
    d = d.dropna(subset=["instr_ord"])
    d["instr_num"] = d["instr_ord"].cat.codes.astype(float)
    d["log_rend"] = np.log1p(d["rend_todos"])
    if len(d) > n_max:
        d = d.sample(n=n_max, random_state=0)
    fig = px.scatter_matrix(
        d,
        dimensions=["idade", "log_rend", "instr_num"],
        color="sexo",
        color_discrete_map={"Homem": "#0072B2", "Mulher": "#D55E00"},
        labels={"idade": "Idade", "log_rend": "log1p(renda)", "instr_num": "Instrucao (ord)", "sexo": "Sexo"},
        title=f"SPLOM — idade, log-renda, instrucao; cor=sexo (n={len(d):,})",
        opacity=0.35,
    )
    fig.update_traces(diagonal_visible=False, showupperhalf=False)
    fig.update_layout(height=700, margin=dict(t=80))
    return fig

fig4 = fig_splom(df)
fig4.show()


## 5. Parallel sets — complementar

### Ligação com o material

Ainda na família de técnicas **baseadas em linha** / fluxos entre eixos, o material trata variantes próximas às coordenadas paralelas para categorias (regiões entre eixos vizinhos indicando relação; cores conforme seleção). **Parallel sets** estendem a ideia: eixos são variáveis categóricas; a largura das faixas (ribbons) é proporcional à frequência conjunta.

### Por que faz sentido na PNAD

Sexo, cor/raça, instrução e condição de ocupação são categóricas centrais na pesquisa. A técnica responde perguntas de composição e fluxo (ex.: como se distribuem ocupados vs não ocupados ao longo da escolaridade e do sexo).

### Julgamento

**Prós:** adequada a tipagem categórica da PNAD; leitura de proporções.  
**Contras:** não mostra magnitude contínua (renda) salvo se discretizada; muitas categorias geram faixas finas.

**Mapeamento abaixo:** sexo → cor/raça → instrução (níveis agregados) → condição de ocupação.


In [15]:
def fig_parallel_sets(df: pd.DataFrame) -> go.Figure:
    d = df.loc[df["sexo"].notna() & df["cor_raca"].notna() & df["instrucao"].notna() & df["cond_ocupacao"].notna()].copy()
    # agrega instrucao em 4 niveis para faixas legíveis
    mapa = {
        "Sem instrucao": "Ate fund.",
        "Fund. incompleto ou equiv.": "Ate fund.",
        "Fund. completo ou equiv.": "Ate fund.",
        "Medio incompleto ou equiv.": "Medio",
        "Medio completo ou equiv.": "Medio",
        "Superior incompleto ou equiv.": "Superior",
        "Superior completo": "Superior",
    }
    d["instr_g"] = d["instrucao"].map(mapa).fillna("Outro")
    d = d.loc[d["cor_raca"].isin(["Branca", "Preta", "Parda"])]
    cols = ["sexo", "cor_raca", "instr_g", "cond_ocupacao"]
    g = d.groupby(cols, as_index=False).size().rename(columns={"size": "n"})

    # constroi nos e links para go.Parcats
    fig = go.Figure(go.Parcats(
        dimensions=[
            dict(label="Sexo", values=g["sexo"]),
            dict(label="Cor/raca", values=g["cor_raca"]),
            dict(label="Instrucao (agreg.)", values=g["instr_g"]),
            dict(label="Cond. ocupacao", values=g["cond_ocupacao"]),
        ],
        counts=g["n"],
        line=dict(color=g["n"], colorscale="Cividis", showscale=True, colorbar=dict(title="n")),
        hoveron="color",
    ))
    fig.update_layout(
        title=f"Parallel sets — sexo → cor/raca → instrucao → ocupacao (n_obs={int(g['n'].sum()):,})",
        height=560, margin=dict(t=80, l=40, r=40, b=40),
    )
    return fig

fig5 = fig_parallel_sets(df)
fig5.show()


## 6. RadViz — complementar

### Ligação com o material

Tópico 04, §1.2 (métodos baseados em força): **RadViz** coloca âncoras (dimensões) em uma circunferência; cada registro equilibra “molas” em direção às âncoras (analogia à lei de Hooke). A posição 2D **não** é um par de variáveis originais — é uma redução que enfatiza proximidade às âncoras de maior valor relativo. O material alerta: a **ordem das âncoras** altera o leiaute.

### Por que faz sentido na PNAD

Permite projetar várias contínuas/ordinais (idade, log-renda, instrução, etc.) num único plano 2D colorido por sexo ou ocupação — útil como vista global de clusters, ao lado do SPLOM.

### Julgamento

**Prós:** está no material; boa para visão de conjunto.  
**Contras:** eixos do plano não têm unidade interpretável (mesma crítica feita ao MDS no slide); overplotting exige amostragem.

**Mapeamento abaixo:** âncoras = idade, log-renda, instrução ordinal (normalizadas 0–1); cor = sexo.


In [16]:
def _radviz_xy(D: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """D: (n, m) valores em [0,1]. Retorna x,y no disco unitario (formula do material)."""
    n, m = D.shape
    angles = np.linspace(0, 2 * np.pi, m, endpoint=False)
    A = np.column_stack([np.cos(angles), np.sin(angles)])  # (m, 2)
    # p = sum(A_j * d_j) / sum(d_j)
    num = D @ A  # (n, 2)
    den = D.sum(axis=1, keepdims=True) + 1e-12
    P = num / den
    return P[:, 0], P[:, 1]


def fig_radviz(df: pd.DataFrame, n_max: int = 5000) -> go.Figure:
    d = df.loc[
        (df["cond_ocupacao"] == "Ocupado")
        & df["rend_todos"].notna() & (df["rend_todos"] > 0)
        & df["idade"].notna() & df["instrucao"].notna() & df["sexo"].notna()
    ].copy()
    d = d.loc[d["rend_todos"] <= d["rend_todos"].quantile(0.99)]
    d["instr_ord"] = pd.Categorical(d["instrucao"], categories=INSTR_ORDER, ordered=True)
    d = d.dropna(subset=["instr_ord"])
    if len(d) > n_max:
        d = d.sample(n=n_max, random_state=0)

    cols = {
        "idade": d["idade"].to_numpy(float),
        "log_rend": np.log1p(d["rend_todos"].to_numpy(float)),
        "instr_num": d["instr_ord"].cat.codes.to_numpy(float),
    }
    # normaliza 0-1 por coluna
    mat = []
    labels = list(cols.keys())
    for k in labels:
        v = cols[k]
        mat.append((v - v.min()) / (v.max() - v.min() + 1e-12))
    D = np.column_stack(mat)
    x, y = _radviz_xy(D)

    # circunferencia e ancoras
    theta = np.linspace(0, 2 * np.pi, 200)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.cos(theta), y=np.sin(theta), mode="lines",
                             line=dict(color="#BBBBBB", width=1), hoverinfo="skip", showlegend=False))
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False)
    fig.add_trace(go.Scatter(
        x=np.cos(angles), y=np.sin(angles), mode="markers+text",
        text=labels, textposition="top center",
        marker=dict(size=10, color="#333333"), showlegend=False, hoverinfo="text",
    ))
    fig.add_trace(go.Scatter(
        x=x, y=y, mode="markers",
        marker=dict(size=5, opacity=0.35,
                    color=(d["sexo"] == "Mulher").astype(int),
                    colorscale=[[0, "#0072B2"], [1, "#D55E00"]],
                    colorbar=dict(title="Sexo", tickvals=[0, 1], ticktext=["Homem", "Mulher"])),
        hovertemplate="sexo=%{customdata}<extra></extra>",
        customdata=d["sexo"],
        name="individuos",
    ))
    fig.update_layout(
        title=f"RadViz — ancoras idade, log-renda, instrucao; cor=sexo (n={len(d):,})",
        xaxis=dict(scaleanchor="y", scaleratio=1, visible=False),
        yaxis=dict(visible=False),
        height=600, margin=dict(t=80, l=40, r=40, b=40),
    )
    return fig

fig6 = fig_radviz(df)
fig6.show()


## Como usar isto no relatório (sem fechar as “3 informações”)

1. Escolher **três** técnicas de famílias diferentes (recomendação: paralelas + treemap + pixels, já no rascunho).  
2. Se o grupo preferir trocar uma delas: SPLOM (ponto) ou parallel sets (fluxo categórico) têm encaixe direto no Tópico 04.  
3. Em cada seção “Informações obtidas”: mapear canais → variáveis; citar a família Ward; justificar com a figura Plotly deste notebook.  
4. RadViz / SPLOM podem ir a apêndice ou “técnicas consideradas”.

*Conteúdo, experimentos e conclusões são meus; a formatação do texto teve assistência de IA.*
